In [2]:
import json
import pandas as pd
import psycopg2

with open('config/logs_db_creds.json') as json_file:
    creds = json.load(json_file)

DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "logs"
DB_USER = creds["user"]
DB_PASSWORD = creds["password"]

logs = []

df = None

try:
    with psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    ) as conn:
        with conn.cursor() as cursor:
            cursor.execute(
                "SELECT * FROM submissions WHERE date > '2025-06-01';")
            logs = cursor.fetchall()

            df = pd.DataFrame(logs, columns=[desc[0]
                              for desc in cursor.description])

except Exception as e:
    print("Error:", e)

In [3]:
games_df = None

try:
    with psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    ) as conn:
        with conn.cursor() as cursor:
            cursor.execute("SELECT * FROM battleground_games;")
            logs = cursor.fetchall()

            games_df = pd.DataFrame(
                logs, columns=[desc[0] for desc in cursor.description])

except Exception as e:
    print("Error:", e)

In [ ]:
df['game_id'] = df['game_id'].astype(int)
df['objective'] = df['objective'].astype(int)

games_df['game_id'] = games_df['game_id'].astype(int)
games_df['id'] = games_df['id'].astype(int)

games_df = games_df.sort_values(by=['class_alias', 'attacker', 'defender'])

games_df = games_df.loc[~games_df.duplicated(
    subset=['class_alias', 'attacker', 'defender'], keep='last')]

games_df = games_df.sort_values(by='date')

Main

In [5]:
import json
import os

setups = []
for file in os.listdir('resources/setups'):
    if file.endswith('.json'):
        with open(os.path.join('resources/setups', file), 'r') as f:
            data = f.read()
            setup = json.loads(data)

        setups.append((int(file.split('.')[0]), setup))

In [6]:
setups_by_class_alias = {}
for id, setup in setups:
    class_alias = setup['class_alias']
    if class_alias not in setups_by_class_alias:
        setups_by_class_alias[class_alias] = []
    setups_by_class_alias[class_alias].append((id, setup))

In [ ]:
import concurrent.futures as cf
import subprocess
import time


def run(cmd):
    process = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )

    stdout_lines = []
    stderr_lines = []

    for line in iter(process.stdout.readline, ''):
        line = line.strip()
        if line:
            current_time = time.strftime("%H:%M:%S", time.gmtime(time.time()))
            print(f"[{current_time} {cmd}] {line}")
            stdout_lines.append(line)

    process.wait()
    stderr = process.stderr.read()
    if stderr:
        stderr_lines = stderr.strip().split('\n')
        for line in stderr_lines:
            print(f"[{cmd}] ERROR: {line}")

    class Result:
        def __init__(self, args, returncode, stdout, stderr):
            self.args = args
            self.returncode = returncode
            self.stdout = '\n'.join(stdout_lines)
            self.stderr = stderr

    return Result(cmd, process.returncode, '\n'.join(stdout_lines), stderr)

repetitions = 1
workers = 10

for i in range(repetitions):
    with cf.ThreadPoolExecutor(max_workers=workers) as pool:
        futures = []
        for class_alias, class_setups in setups_by_class_alias.items():
            for id, setup in class_setups:
                if ((games_df['attacker'] == setup['attacker_model_name']) & (games_df['defender'] == setup['defender_model_name']) & (games_df['class_alias'] == class_alias)).sum() > i:
                    continue

                print(f"Submitting game with attacker {setup['attacker_model_name']} and defender {setup['defender_model_name']}. Class {class_alias}. Setup ID {id}.")
                futures.append(
                    pool.submit(
                        run,
                        f"python game.py --id {id}"))

        for fut in cf.as_completed(futures):
            res = fut.result()
            print(f"######################## {res.args} READY ########################")